In [16]:
import json #import the json library so we can read the JSON files

In [17]:
TEST_ENEMY_LEVEL = 100 
TEST_CHARACTER_LEVEL = 90

In [18]:
def load_json(filepath): # load data from a JSON file
    with open(filepath, "r") as f: # open the file in read mode
        data = json.load(f) # load the JSON data from the file into a Python dictionary
    return data

In [19]:
def apply_defense_multiplier(damage, enemy_level, character_level): # apply the defense multiplier to the base damage based on the enemy's defense and the character's level
    def_multiplier = (character_level + 100) / ((character_level + 100) + (enemy_level +100)) # calculate the defense multiplier using the formula
    return damage * def_multiplier # return the final damage after applying the defense multiplier
def apply_res_multiplier(enemy_data, elemental_dmg_type, damage): 
    res_type = elemental_dmg_type 
    res_value = enemy_data["resistances"][res_type]
    res_multiplier = 1 - res_value
    return damage * res_multiplier
def calculate_base_damage(multiplier,scaling_stat_value): # calculate the base damage of a hit based on its multiplier and the character's scaling stat value
    return (multiplier / 100) * scaling_stat_value # return the base damage by multiplying the multiplier (as a percentage) by the scaling stat value

In [20]:
def get_base_stats_at_level(character_data, character_level): 
    base_stats_by_level = character_data["base_stats_by_level"]
    level_key = str(character_level)
    level_stat = base_stats_by_level[level_key]
    return level_stat

def get_base_stat_list(character_data, level_stat):
    base_stats = character_data["base_stats"].copy()
    base_stats.update(level_stat)
    return base_stats

In [21]:
#---------------------------------------------------------------Takes artifacts, weapon, and character data and merges stats into a single dict----------------------------------------------------
def aggregate_equipment(equipment_list):
    build_flat_totals = {}
    build_percent_totals = {}
    build_bonus_dmg_totals = {}
    for equipment in equipment_list:
        flat_stats = equipment["flat_stat_values"]
        for stat_name in flat_stats:
            build_flat_totals[stat_name] = build_flat_totals.get(stat_name,0) + flat_stats[stat_name]
        percent_values = equipment["percent_stat_values"]
        for stat_name in percent_values:
            build_percent_totals[stat_name] = build_percent_totals.get(stat_name,0) + percent_values[stat_name]
        bonus_dmg_totals = equipment["bonus_dmg_values"]
        for bonus_dmg_name in bonus_dmg_totals:
            build_bonus_dmg_totals[bonus_dmg_name] = build_bonus_dmg_totals.get(bonus_dmg_name,0) + bonus_dmg_totals[bonus_dmg_name]
    build_total = {"flat_stat_values": build_flat_totals, "percent_stat_values": build_percent_totals, "bonus_dmg_values": build_bonus_dmg_totals}
    return build_total
def calculate_flat_stats(base_stats, build_total, weapon_data):
    base_character_stats = base_stats
    weapon_totals = weapon_data["flat_stat_values"]
    flat_totals = {}
    for base_character_stat in base_character_stats:
        base_stat = base_character_stats[base_character_stat]
        weapon_stat = weapon_totals[base_character_stat]
        flat_stat = base_stat + weapon_stat
        flat_totals[base_character_stat] = flat_stat
    flat_max_hp_stat = flat_totals["max_hp"]
    flat_atk_stat = flat_totals["atk"]
    flat_def_stat = flat_totals["def"]
    percent_max_hp_increase = weapon_data["percent_stat_values"].get("max_hp",0) + build_total["percent_stat_values"].get("max_hp",0) 
    percent_atk_increase = weapon_data["percent_stat_values"].get("atk",0) + build_total["percent_stat_values"].get("atk",0) 
    percent_def_increase = weapon_data["percent_stat_values"].get("def",0) + build_total["percent_stat_values"].get("def",0)
    final_max_hp_value = flat_max_hp_stat * (percent_max_hp_increase + 1)
    final_atk_value = flat_atk_stat * (percent_atk_increase + 1)
    final_def_value = flat_def_stat * (percent_def_increase + 1)
    flat_totals["max_hp"] = final_max_hp_value
    flat_totals["atk"] = final_atk_value
    flat_totals["def"] = final_def_value
    return flat_totals
def calculate_bonus_stats(character_data, build_total, weapon_data):
    character_bonus_stats = character_data["bonus_dmg_values"]
    weapon_bonus_stats = weapon_data["bonus_dmg_values"]
    build_bonus_stats = build_total["bonus_dmg_values"]
    bonus_totals = {}
    for bonus_stat in character_bonus_stats:
        character_stat = character_bonus_stats[bonus_stat]
        weapon_stat = weapon_bonus_stats[bonus_stat]
        build_stat = build_bonus_stats[bonus_stat]
        bonus_stats = character_stat + weapon_stat + build_stat
        bonus_totals[bonus_stat] = bonus_stats
    return bonus_totals
def calculate_final_stats(build_total, flat_totals, bonus_totals):
    build_totals = build_total["flat_stat_values"]
    stat_total = {}
    for total in build_totals:
        build_stat = build_totals[total]
        stat_total[total] = flat_totals[total] + build_stat
    stat_total.update(bonus_totals)
    return stat_total

In [22]:
def get_normal_attack_multiplier(character_data, hit_number, talent_level): # get the multiplier for a specific hit of the normal attack at a given talent level
    hits = character_data["talents"]["normal_attack"]["hits"] # get the list of hits for the normal attack
    for hit in hits: # iterate through each hit in the list
        if hit["hit_number"] == hit_number: # check if the hit number matches the one we're looking for
            level_lookup = hit["multiplier_by_level"]   # open the folder — get all 11 sticky notes
            level_key = str(talent_level)               # translate 9 (numeral) into "9" (word), so it matches the labels
            multiplier = level_lookup[level_key]        # find the sticky note labeled "9", read its number
            return multiplier                           # hand that number back
    return None
def calculate_hit_damage(character_data, compiled_stats, hit_number, enemy_level, character_level, talent_level, enemy_data): # calculate the final damage of a specific hit of the normal attack, taking into account the character's data, the hit number, the enemy's defense, and the character's level
    multiplier = get_normal_attack_multiplier(character_data, hit_number, talent_level) # get the multiplier for the specified hit number and talent level
    scaling_stat_name = character_data["talents"]["normal_attack"]["scaling_stat"] # get the name of the scaling stat used for the normal attack
    scaling_stat_value = compiled_stats[scaling_stat_name] # get the value of the scaling stat from the character's base stats
    elemental_bonus = character_data["talents"]["normal_attack"]["elemental_dmg_type"]
    elemental_dmg_value = compiled_stats[elemental_bonus]
    other_dmg_value = compiled_stats.get("other",0)
    bonus_dmg = elemental_dmg_value + other_dmg_value
    base_dmg = calculate_base_damage(multiplier, scaling_stat_value) # calculate the base damage using the multiplier and scaling stat value
    normal_attack_dmg = base_dmg * (bonus_dmg + 1)
    defense_applied = apply_defense_multiplier(normal_attack_dmg, enemy_level, character_level) # apply the defense multiplier to the base damage to get the final damage
    final = apply_res_multiplier(enemy_data, elemental_bonus, defense_applied)
    return final # return the final damage value

In [23]:
def get_elemental_skill_multiplier(character_data, crystal_shrapnel_stacks): 
        stacks = character_data["talents"]["elemental_skill"]["stacks"] 
        for stack in stacks:
            if stack["stacks_consumed"] == crystal_shrapnel_stacks:
                shardshots = stack["shardshots"]
                dmg_bonus = stack["dmg_bonus"]
                pct_of_base = stack["pct_of_base"]
                return shardshots, dmg_bonus, pct_of_base
def get_elemental_skill_base_multiplier(character_data, talent_level):
    base_multiplier_by_level = character_data["talents"]["elemental_skill"]["base_multiplier_by_level"]
    multiplier_key = str(talent_level)
    base_multiplier = base_multiplier_by_level[multiplier_key]
    return base_multiplier
def calculate_skill_damage(character_data, crystal_shrapnel_stacks, talent_level, enemy_level, character_level, compiled_stats, enemy_data):
    base_multiplier = get_elemental_skill_base_multiplier(character_data, talent_level)
    shardshots, dmg_bonus, pct_of_base = get_elemental_skill_multiplier(character_data, crystal_shrapnel_stacks)
    
    scaled_multiplier = base_multiplier * (pct_of_base / 100)
    final_multiplier = scaled_multiplier * (1 + dmg_bonus / 100)
    
    scaling_stat_name = character_data["talents"]["elemental_skill"]["scaling_stat"]
    scaling_stat_value = compiled_stats[scaling_stat_name]

    elemental_bonus = character_data["talents"]["elemental_skill"]["elemental_dmg_type"]
    elemental_dmg_value = compiled_stats[elemental_bonus]
    other_dmg_value = compiled_stats.get("other",0)
    bonus_dmg = elemental_dmg_value + other_dmg_value
    base_dmg = calculate_base_damage(final_multiplier, scaling_stat_value)
    elemental_skill_dmg = base_dmg * (bonus_dmg + 1)
    defense_applied = apply_defense_multiplier(elemental_skill_dmg, enemy_level, character_level)
    final = apply_res_multiplier(enemy_data, elemental_bonus, defense_applied)
    return final

In [24]:
def calculate_burst_damage(character_data, talent_level, enemy_level, character_level, compiled_stats, enemy_data):
    scaling_stat_name = character_data["talents"]["elemental_burst"]["scaling_stat"]
    scaling_stat_value = compiled_stats[scaling_stat_name]

    skill_multiplier = character_data["talents"]["elemental_burst"]["skill_dmg_by_level"][str(talent_level)]
    cannon_multiplier = character_data["talents"]["elemental_burst"]["cannon_fire_support_dmg_by_level"][str(talent_level)]
    hit_count = character_data["talents"]["elemental_burst"]["cannon_fire_support_hit_count"]

    elemental_bonus = character_data["talents"]["elemental_burst"]["elemental_dmg_type"]
    elemental_dmg_value = compiled_stats[elemental_bonus]
    other_dmg_value = compiled_stats.get("other",0)
    bonus_dmg = elemental_dmg_value + other_dmg_value
    skill_base_damage = calculate_base_damage(skill_multiplier, scaling_stat_value)
    skill_dmg = skill_base_damage * (bonus_dmg + 1)
    skill_defense_applied = apply_defense_multiplier(skill_dmg, enemy_level, character_level)
    skill_final = apply_res_multiplier(enemy_data, elemental_bonus, skill_defense_applied)
    
    cannon_base_damage = calculate_base_damage(cannon_multiplier, scaling_stat_value)
    cannon_dmg = cannon_base_damage * (bonus_dmg + 1)
    cannon_defense_applied_single_hit = apply_defense_multiplier(cannon_dmg, enemy_level, character_level)
    cannon_final_single_hit = apply_res_multiplier(enemy_data, elemental_bonus, cannon_defense_applied_single_hit)
    cannon_final_total = cannon_final_single_hit * hit_count
    
    total_damage = skill_final + cannon_final_total
    
    return total_damage, skill_final, cannon_final_single_hit, cannon_final_total

In [26]:
#-------------------------------------------------------------------------------Compiles Build-------------------------------------------------------------------------------------
character_data = load_json("naviav5.json")
weapon_data = load_json("verdictv2.json")
flower_data = load_json("selfless_floral_accessory.json")
feather_data = load_json("honest_quill.json")
sands_data = load_json("faithful_hourglass.json")
goblet_data = load_json("a_horn_unwindedv2.json")
circlet_data = load_json("compassionate_ladies_hat.json")
equipment_list = [flower_data, feather_data, sands_data, goblet_data, circlet_data]
build_data = aggregate_equipment(equipment_list)
level_stat = get_base_stats_at_level(character_data, TEST_CHARACTER_LEVEL)
base_stats = get_base_stat_list(character_data, level_stat)
flat_totals = calculate_flat_stats(base_stats, build_data, weapon_data)
bonus_totals = calculate_bonus_stats(character_data, build_data, weapon_data)
result = calculate_final_stats(build_data, flat_totals, bonus_totals)
print(result)

{'max_hp': 18386.03, 'atk': 2309.0560299999997, 'def': 926.5006000000001, 'max_stamina': 240, 'elemental_mastery': 0, 'crit_rate': 88.1, 'crit_dmg': 160.70000000000002, 'energy_recharge': 140.2, 'cd_reduction': 0.0, 'shield_strength': 0.0, 'pyro': 0.0, 'hydro': 0.0, 'dendro': 0.0, 'electro': 0.0, 'anemo': 0.0, 'cryo': 0.0, 'geo': 0.466, 'physical': 0.0, 'other': 0.0}


In [27]:
#-------------------------------------------------------------------------------Calculates 1-hit dmg for a talent level-------------------------------------------------------------------------------------

character_data = load_json("naviav5.json")
enemy_data = load_json("testdummy.json")
compiled_stats = calculate_final_stats(build_data, flat_totals, bonus_totals)
result = calculate_hit_damage(enemy_data=enemy_data, character_data=character_data, compiled_stats=compiled_stats, hit_number="1", talent_level=1, enemy_level=TEST_ENEMY_LEVEL, character_level=TEST_CHARACTER_LEVEL) # calls by specific hit number and talent level
print(result)

946.8266489045538


In [28]:
#-------------------------------------------------------------------------------Calculates NA Combo Damage------------------------------------------------------------------------------------

character_data = load_json("naviav5.json")
enemy_data = load_json("testdummy.json")
hit_number = character_data["talents"]["normal_attack"]["hits"] # get the list of hits for the normal attack
total = 0 # initialize a variable to keep track of the total damage
for hit in hit_number: # iterate through each hit in the list
        result = calculate_hit_damage(enemy_data=enemy_data, character_data=character_data, hit_number=hit["hit_number"], talent_level=10, enemy_level=TEST_ENEMY_LEVEL, character_level=TEST_CHARACTER_LEVEL, compiled_stats=compiled_stats) # calls full combo by talent level
        total = total + result
print(total)

8367.752623178076


In [29]:
character_data = load_json("naviav5.json")
result = get_elemental_skill_multiplier(character_data=character_data, crystal_shrapnel_stacks=6)
print(result) # prints shardshots, dmg_bonus

(11, 45, 200)


In [30]:
character_data = load_json("naviav5.json")
base = get_elemental_skill_base_multiplier(character_data=character_data, talent_level=13)
print(base)

838.95


In [31]:
#-------------------------------------------------------------------------------Calculates Skill DMG (Navia Specific)-------------------------------------------------------------------------------------
character_data = load_json("naviav5.json")
enemy_data = load_json("testdummy.json")
result = calculate_skill_damage(enemy_data=enemy_data, character_data=character_data, compiled_stats=compiled_stats, crystal_shrapnel_stacks=6, talent_level=13, enemy_level=TEST_ENEMY_LEVEL, character_level=TEST_CHARACTER_LEVEL)
print(result)

36110.54318832825


In [32]:
#-------------------------------------------------------------------------------Calculates Burst DMG (Navia Specific)-------------------------------------------------------------------------------------
character_data = load_json("naviav5.json")
enemy_data = load_json("testdummy.json")
result = calculate_burst_damage(enemy_data=enemy_data, character_data=character_data, compiled_stats=compiled_stats, talent_level=13, enemy_level=TEST_ENEMY_LEVEL, character_level=TEST_CHARACTER_LEVEL)
print(result)

(6454.452267431295, 2371.7926560478327, 1360.8865371278207, 4082.659611383462)
